# Orin Nano R35.5.0: SFace inference and masking inside OP-TEE

**Both inference and masking execute in the trusted application.** This notebook only loads an aligned face crop and public model weights, starts measurements, and receives a masked vector and timing counters. There is no software fallback.

Build/deploy `secure_sface` using `README.md` first. The trusted CPU implementation is distinct from normal-world OpenCV/CUDA. Image acquisition and landmark alignment remain outside the TEE; the input image is still visible to Linux. No raw embedding, activation or mask is returned by the TA.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone

ROOT = Path.cwd()
if not (ROOT / "secure_benchmark.py").exists():
    ROOT = ROOT / "jetson_benchmark"
if not (ROOT / "secure_benchmark.py").exists():
    raise RuntimeError("Open this notebook from jetson_benchmark or its parent directory")
sys.path.insert(0, str(ROOT.resolve()))
from secure_benchmark import run_secure_benchmark
from benchmark import device_info

device_info()

## Settings

Use an already landmark-aligned 112×112 face crop. Merely resizing an image is not alignment. The TA converts the BGR pixels to RGB/NCHW and executes the model's normalization, all neural-network operators, fixed-point encoding, and fresh additive masking.

`weights.bin`, `manifest.json`, the bridge and the installed TA must come from the same model export. The TA verifies every weight block before use. Consult the original SFace checkpoint provenance before claiming reproduction of the old paper.

In [ ]:
IMAGE = ROOT / "images/aligned_face.png"
GENERATED = ROOT / "secure_sface/generated"
LIBRARY = ROOT / "secure_sface/host/libsecure_sface.so"
ITERATIONS = 10
SECONDS = 30
WARMUP = 2
MEASURE_POWER = True
POWER_RAIL = "VDD_IN"
TEGRASTATS = "tegrastats"
INTERVAL_MS = 100
NOTES = "Orin Nano; L4T R35.5.0. Add nvpmodel mode, clock policy, cooling and background workloads."

## Measure

The high-resolution host timer covers the whole trusted pipeline, including transfer/hash verification of streamed weights, TEE transitions and buffer clearing. TA phase timers use millisecond counters; a zero for masking means it was below the counter resolution, not free.

Power is integrated for the selected Jetson rail over the repeated run. It includes idle power and host/TEE overhead. `MEASURE_POWER=False` disables only energy measurement; inference and masking still require real OP-TEE. Warmup, file loading, alignment, opening the session and result export are excluded.

In [ ]:
OUTPUT = ROOT / "results" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
summary = run_secure_benchmark(
    IMAGE, output=OUTPUT, generated=GENERATED, library=LIBRARY,
    iterations=ITERATIONS, seconds=SECONDS, warmup=WARMUP,
    power=MEASURE_POWER, rail=POWER_RAIL, tegrastats=TEGRASTATS,
    interval_ms=INTERVAL_MS, notes=NOTES,
)
summary

## Export outside the TEE

The last masked vector is written as 128 decimal uint64 strings. Pads are discarded inside the TA, so this benchmark output is not yet a replacement for Obscura's correlated dealer material. The output directory also contains timing CSV, summary JSON and power samples when enabled.

In [ ]:
import json
packet = json.loads((OUTPUT / "masked_embedding.json").read_text())
assert summary["inference_inside_tee"] and summary["masking_inside_tee"]
assert len(packet["values"]) == 128
assert all(0 <= int(v) < 2**64 for v in packet["values"])
print("Results:", OUTPUT)
print("Timing (ms):", summary["timings_ms"])
print("Energy:", summary["power"])
print("Masked vector saved:", OUTPUT / "masked_embedding.json")